# Example of Bayesian inference for the self-discharge count of a thyratron

Copyright (C) 2024 Hirokazu Maesaka, RIKEN SPring-8 Center. All rights reserved.

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from scipy.stats import gamma, nbinom
plt.rcParams['font.size'] = 16
%matplotlib inline

# Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
basedir = '/content/drive/MyDrive/Colab Notebooks/ISBA24'
data = pd.read_csv(f"{basedir}/thyratron_selfdischarge.csv")
display(data)

In [ ]:
# Plot
fig, ax = plt.subplots(2, 1, figsize=(6,6))
ax[0].plot(data["hrs"], data["cnt"])
ax[0].set_xlabel("Hours")
ax[0].set_ylabel("Counts")
ax[0].set_title("CB08-4 Self-discharge Counts in 2021")
ax[0].grid()
ax[1].plot(data["hrs"], data["diff"])
ax[1].set_xlabel("Hours")
ax[1].set_ylabel("Counts/hour")
ax[1].grid()
plt.tight_layout()
plt.show()

# Bayesian inference of the self-discharge counts per hour
- Assume Poisson distribution.
  - ${\rm Poi}(n|\lambda) = \frac{\lambda^n}{n!} e^{-\lambda}$
  - $E(n) = V(n) = \lambda$
- The conjugate prior function of the Poisson distribution is the Gamma distribution.
  - ${\rm Gam}(\lambda|\alpha,\beta) = \frac{\beta^\alpha}{\Gamma(\alpha)} \lambda^{\alpha-1} e^{-\beta\lambda},
    \qquad \alpha > 0, \qquad \beta > 0$
  - $E(\lambda) = \frac{\alpha}{\beta}$
  - $\ln {\rm Gam}(\lambda|\alpha,\beta) = (\alpha - 1) \ln \lambda - \beta \lambda + {\rm const.}$
- If we observe the data ${\bf N} = \{n_1, n_2, \cdots, n_T\}$, the posterior distribution is calculated to be
$$ p(\lambda|{\bf N}) \propto p({\bf N}|\lambda) p(\lambda)
= \left\{ \prod_{t=1}^T {\rm Poi}(n_t|\lambda) \right\} {\rm Gam}(\lambda|\alpha,\beta). $$
- The logarithm of the distribution is
$$ \begin{align}
\ln p(\lambda|{\bf N}) &= \sum_{t=1}^T \left[ n_t \ln \lambda - \ln n_t! - \lambda \right]
+ (\alpha -1)\ln \lambda - \beta\lambda + \alpha \ln \beta - \ln \Gamma(\alpha) + {\rm const.} \\
&= \left( \sum_{t=1}^T n_t + \alpha - 1 \right) \ln \lambda - (T + \beta)\lambda + {\rm const.}
\end{align} $$
- This formula is the same as the Gamma distribution. Therefore,
$$ p(\lambda|{\bf N}) = {\rm Gam}(\lambda|\hat{\alpha},\hat{\beta}), \qquad
\hat{\alpha} = \sum_{t=1}^T n_t + \alpha, \qquad \hat{\beta} = T+\beta. $$

- Now, we use the mean of the first 100 hours for the prior mean.
- We set the initial $\alpha$ to 1.

In [ ]:
# Prior distribution of lambda
hrs0 = 100
a0 = 1.
b0 = a0 / data["diff"][data["hrs"]<hrs0].mean()
prior = gamma(a0, scale=1./b0)
print(f"alpha = {a0}, beta = {b0}")

x = np.linspace(prior.ppf(0.), prior.ppf(0.99), 1000)
fig, ax = plt.subplots()
ax.plot(x, prior.pdf(x))
ax.grid()
ax.set_xlabel("$\lambda$")
ax.set_title("Prior Distribution")
plt.show()

In [ ]:
# Posterior parameters
data["a"] = data["cnt"] + a0
data["b"] = data["hrs"] + b0
display(data)

In [ ]:
# Posterior distribution at every 100 hours
step = 100
for hrs in range(step, data.index.max(), step):
    param = data.loc[hrs]
    post = gamma(param["a"], scale=1./param["b"])
    fig, ax = plt.subplots()
    ax.plot(x, post.pdf(x))
    ax.grid()
    ax.set_title(f"Posterior after {hrs} hours")
    ax.set_xlabel("$\lambda$")
    plt.show()
    print(f"time = {hrs} hours, alpha = {param['a']}, beta = {param['b']}")

In [ ]:
# Evolution of the expectation of lambda
data["E"] = data["a"] / data["b"]
display(data)
fig, ax = plt.subplots()
ax.plot(data["hrs"], data["E"])
ax.set_xlabel("Hours")
ax.set_ylabel("$E(\lambda)$")
ax.grid()
plt.show()

In [ ]:
# with 10% - 90% interval
low = np.empty(len(data))
high = np.empty(len(data))
for i in range(len(data)):
    post = gamma(data["a"].iloc[i], scale=1./data["b"].iloc[i])
    low[i] = post.ppf(0.1)
    high[i] = post.ppf(0.9)
data["low"] = low
data["high"] = high
display(data)
fig, ax = plt.subplots()
ax.plot(data["hrs"], data["E"], label="$E(\lambda)$")
ax.plot(data["hrs"], data["low"], '--', label="lower bound")
ax.plot(data["hrs"], data["high"], '--', label="upper bound")
ax.set_xlabel("Hours")
ax.set_ylabel("$\lambda$")
ax.grid()
ax.legend()
plt.show()

# Predicted distribution of the count
- $n_*$: predicted counts
$$ \begin{align}
p(n_*) &= \int p(n_*|\lambda)p(\lambda)d\lambda \\
&= \int {\rm Poi}(n_*|\lambda) {\rm Gam}(\lambda|\hat{\alpha},\hat{\beta}) d\lambda \\
&= \frac{\hat{\beta}^{\hat{\alpha}}}{n_*! \Gamma(\hat{\alpha})}
   \int \lambda^{n_*+\hat{\alpha}-1} e^{-(\hat{\beta}+1)\lambda} d\lambda \\
&= \frac{\hat{\beta}^{\hat{\alpha}} \Gamma(n_*+\hat{\alpha})}
        {n_*! (\hat{\beta}+1)^{n_*+\hat{\alpha}} \Gamma(\hat{\alpha})} \\
&= \frac{\Gamma(n_*+r)}{n_*! \Gamma(r)} (1-p)^r p^{n_*}
\end{align} $$
- Here, we defined $r=\hat{\alpha}$, $p=\frac{1}{\hat{\beta}+1}$.
- This distribution is called negative binomial distribution.
- $E(n_*) = \frac{pr}{1-p} = \frac{\hat{\alpha}}{\hat{\beta}}$
- $E(n_*^2) = \frac{pr(pr+1)}{(1-p)^2} = \frac{\hat{\alpha}(\hat{\alpha}+\hat{\beta}+1)}{\hat{\beta}^2}$
- $V(n_*) = E(n_*^2) - [E(n_*)]^2 = \frac{pr}{(1-p)^2} = \frac{\hat{\alpha}(\hat{\beta}+1)}{\hat{\beta}^2}$

In [ ]:
# Predicted distribution at every 100 hours
step = 100
n = np.arange(0, 6, dtype=int)
for hrs in range(step, data.index.max(), step):
    param = data.loc[hrs]
    pred = nbinom(param["a"], param["b"]/(1+param["b"]))
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    for i in range(2):
        ax[i].vlines(n, 0, pred.pmf(n))
        ax[i].plot(n, pred.pmf(n), 'o')
        ax[i].grid()
        ax[i].set_title(f"Prediction after {hrs} hours")
        ax[i].set_xlabel("$n_*$")
    ax[0].set_ylim([None, 1.0])
    ax[1].set_yscale("log")
    ax[1].set_ylim([None, 10.0])
    plt.show()

# Expectation of predicted cumulative counts

In [ ]:
data["pred"] = data["hrs"]*data["E"]
data["dcnt"] = data["cnt"]-data["pred"]
display(data)

In [ ]:
# Plot
fig, ax = plt.subplots(2, 1, figsize=(6,6))
ax[0].plot(data["hrs"], data["cnt"], label="Data")
ax[0].plot(data["hrs"], data["pred"], label="Expectation")
ax[0].grid()
ax[0].legend()
ax[0].set_xlabel("Hours")
ax[0].set_ylabel("Counts")
ax[1].plot(data["hrs"], data["dcnt"])
ax[1].grid()
ax[1].set_xlabel("Hours")
ax[1].set_ylabel("Difference")
plt.tight_layout()
plt.show()

In [ ]:
display(data[(data["hrs"]>100)&(data["dcnt"]>1)])

- The difference of the cumulative counts exceeds 1 at 1658 hours.
- This is about 5 days before the death.

# Distribution of predicted cumulative counts
- The distribution of the sum of the probability variable $n_*$ must be calculated.
- It can be calculated by using the moment-generating function (MGF).
  - ${\rm MGF}_{X}(t) = E(e^{tX}) = \int e^{tX} p(X) dX, \qquad t\in{\mathbb R}, \qquad X$: probability variable.
  - MGF is something like a Laplace transform of a probability distribution.
  - Usually, the distribution of the sum of probability variables can be calculating by taking the convolution of these variables.
  - The Laplace transform of a convolution is a product, so the sum of probability variables can be transformed to a product.
  - ${\rm MGF}_{X+Y}(t) = {\rm MGF}_{X}(t) {\rm MGF}_{Y}(t)$
- The MGF of a negative binomial distribution is
  ${\rm MGF}_{n_*} = \left( \frac{1-p}{1-pe^t} \right)^r = \left( \frac{\hat{\beta}}{\hat{\beta}+1-e^t} \right)^{\hat{\alpha}}$.
- The MGF of the sum of the negative binomial distribution is
  ${\rm MGF}_{T n_*} = \left( \frac{1-p}{1-pe^t} \right)^{Tr} = \left( \frac{\hat{\beta}}{\hat{\beta}+1-e^t} \right)^{T\hat{\alpha}}$.
- Therefore, the distribution of predicted cumulative counts is also a negative binomial distribution.
  $r=T\hat{\alpha}$, $p=\frac{1}{\hat{\beta}+1}$

In [ ]:
# Distribution of the predicted cumulative counts at every 100 hours
step = 100
for hrs in range(step, data.index.max(), step):
    param = data.loc[hrs]
    pred = nbinom(param["a"]*hrs, param["b"]/(1+param["b"]))
    nmin = pred.ppf(0.001)
    nmax = pred.ppf(0.999)
    n = np.arange(nmin, nmax+1, dtype=int)
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    for i in range(2):
        ax[i].vlines(n, 0, pred.pmf(n))
        ax[i].plot(n, pred.pmf(n), 'o', label="Prediction")
        if i == 1:
            ax[1].set_yscale("log")
        ylim = ax[i].get_ylim()
        ax[i].vlines(param["cnt"], ylim[0], ylim[1], "C1", label="Data")
        ax[i].set_ylim(ylim)
        ax[i].grid()
        ax[i].set_title(f"Cumulative counts after {hrs} hours")
        ax[i].set_xlabel("$\sum n_*$")
        ax[i].legend()
    plt.show()

In [ ]:
# 10% - 90% interval
low = np.empty(len(data))
high = np.empty(len(data))
for i in range(len(data)):
    pred = nbinom(data["a"].iloc[i]*i, data["b"].iloc[i]/(1+data["b"].iloc[i]))
    low[i] = pred.ppf(0.1)
    high[i] = pred.ppf(0.9)
data["low2"] = low
data["high2"] = high
display(data)
fig, ax = plt.subplots(2, 1, figsize=(6, 8))
ax[0].plot(data["hrs"], data["pred"], label="prediction")
ax[0].plot(data["hrs"], data["low2"], '--', label="lower bound")
ax[0].plot(data["hrs"], data["high2"], '--', label="upper bound")
ax[0].set_xlabel("Hours")
ax[0].set_ylabel("$\sum n_*$")
ax[0].grid()
ax[0].legend()
ax[1].plot(data["hrs"], data["cnt"]-data["pred"], label="data-prediction")
ax[1].plot(data["hrs"], data["low2"]-data["pred"], '--', label="lower bound")
ax[1].plot(data["hrs"], data["high2"]-data["pred"], '--', label="upper bound")
ax[1].set_xlabel("Hours")
ax[1].set_ylabel("$\sum n - \sum n_*$")
ax[1].grid()
ax[1].legend()
fig.tight_layout()
plt.show()

- The data is well within the predicted distribution.
- We need to compare the data with the past distribution (24 hours before etc.).

In [ ]:
# Prediction of the cumulative counts from the data 24 hours before
thre = 0.05
deltat = 24
expect = np.zeros(len(data))
low = np.zeros(len(data))
high = np.zeros(len(data))
for i in range(deltat, len(data)):
    param = data.iloc[i-deltat]
    pred = nbinom(param["a"]*i, param["b"]/(1+param["b"]))
    expect[i] = pred.mean()
    low[i] = pred.ppf(thre)
    high[i] = pred.ppf(1.-thre)
data["pred3"] = expect
data["low3"] = low
data["high3"] = high
display(data)
fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(data["hrs"], data["pred3"], label="prediction")
ax[0].plot(data["hrs"], data["low3"], '--', label="lower bound")
ax[0].plot(data["hrs"], data["high3"], '--', label="upper bound")
ax[0].set_xlabel("Hours")
ax[0].set_ylabel("$\sum n_*(-\Delta t)$")
ax[0].grid()
ax[0].legend()
ax[1].plot(data["hrs"], data["cnt"]-data["pred3"], label="data-prediction")
ax[1].plot(data["hrs"], data["low3"]-data["pred3"], '--', label="lower bound")
ax[1].plot(data["hrs"], data["high3"]-data["pred3"], '--', label="upper bound")
ax[1].set_xlabel("Hours")
ax[1].set_ylabel("$\sum n - \sum n_*(-\Delta t)$")
ax[1].grid()
ax[1].legend()
plt.show()

In [ ]:
# Prediction of the cumulative counts
fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(data["hrs"], data["cnt"], label="data")
ax[0].plot(data["hrs"], data["low3"], '--', label="lower bound")
ax[0].plot(data["hrs"], data["high3"], '--', label="upper bound")
ax[0].set_xlabel("Hours")
ax[0].set_ylabel("$\sum n$")
ax[0].grid()
ax[0].legend()
ax[1].plot(data["hrs"], data["cnt"]-data["pred3"], label="data-prediction")
ax[1].plot(data["hrs"], data["low3"]-data["pred3"], '--', label="lower bound")
ax[1].plot(data["hrs"], data["high3"]-data["pred3"], '--', label="upper bound")
ax[1].set_xlabel("Hours")
ax[1].set_ylabel("$\sum n - \sum n_*(-\Delta t)$")
ax[1].grid()
ax[1].legend()
plt.show()

In [ ]:
display(data[(data["hrs"]>100)&(data["cnt"]>data["high3"])])

- We set the upper limit to the 95% limit of the distribution 24 hours before.
- The difference of the cumulative counts exceeds this limit at 1616 hours.
- This is about 7 days before the death.

In [ ]:
# Distribution of the predicted cumulative counts from the data 24 hours before
step = 100
for hrs in range(step, data.index.max(), step):
    param = data.loc[hrs-deltat]
    pred = nbinom(param["a"]*hrs, param["b"]/(1+param["b"]))
    nmin = pred.ppf(0.001)
    nmax = pred.ppf(0.999)
    n = np.arange(nmin, nmax+1, dtype=int)
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    for i in range(2):
        ax[i].vlines(n, 0, pred.pmf(n))
        ax[i].plot(n, pred.pmf(n), 'o', label="Prediction")
        if i == 1:
            ax[1].set_yscale("log")
        ylim = ax[i].get_ylim()
        ax[i].vlines(data["cnt"].loc[hrs], ylim[0], ylim[1], "C1", label="Data")
        ax[i].set_ylim(ylim)
        ax[i].grid()
        ax[i].set_title(f"Cumulative counts after {hrs} hours")
        ax[i].set_xlabel("$\sum n_*$")
        ax[i].legend()
    plt.show()